## 1. run the next two cells as-is

In [1]:
#import libraries and set some variables

import requests
import pandas as pd
import xml.etree.ElementTree as ET
from pprint import pprint
from urllib.parse import quote
import numpy as np
import urllib.parse
import os
import warnings
from rapidfuzz import process, fuzz
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

#nltk.download('punkt_tab')
#nltk.download('stopwords')

pd.set_option('display.max_columns', None)


#Setting a few Alma Analytics API variables and whatnot
base_path = (r'https://api-na.hosted.exlibrisgroup.com/almaws/v1/analytics/reports?'
             r'limit=1000'
             r'&apikey=l8xxe4f2dc8ebe644d88a84c2ab98b67e6e5'
             r'&path=')  

token_path = (r'https://api-na.hosted.exlibrisgroup.com/almaws/v1/analytics/reports?'
             r'limit=1000'
             r'&apikey=l8xxe4f2dc8ebe644d88a84c2ab98b67e6e5'
             r'&token=')

loc1 = 'shared'
loc2 = 'University of Tennessee, Knoxville'
loc3 = 'Reports'
loc4 = 'Collections'
loc5 = 'Textbook ISBNs for Overlap Report_TM'
if loc5 != '':
    location = '/' + loc1 + '/' + loc2 + '/' + loc3 + '/' + loc4 + '/' + loc5 
else:
    location = '/' + loc1 + '/' + loc2 + '/' + loc3 + '/' + loc4 + '/'

get_path = location 
url_encoded_path = quote(get_path)
report_path = base_path + url_encoded_path

xml_snippet = '''
<sawx:expr xsi:type="sawx:list" op="in"
  xmlns:saw="com.siebel.analytics.web/report/v1.1"
  xmlns:sawx="com.siebel.analytics.web/expression/v1.1" 
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
  xmlns:xsd="http://www.w3.org/2001/XMLSchema">

<sawx:expr xsi:type="sawx:sqlExpression">"Bibliographic Details"."MMS Id"</sawx:expr>
{}
</sawx:expr>
'''

In [2]:
report_path

'https://api-na.hosted.exlibrisgroup.com/almaws/v1/analytics/reports?limit=1000&apikey=l8xxe4f2dc8ebe644d88a84c2ab98b67e6e5&path=/shared/University%20of%20Tennessee%2C%20Knoxville/Reports/Collections/Textbook%20ISBNs%20for%20Overlap%20Report_TM'

In [3]:
#define some functions

#function to extract data from the analytics report
def extract_data(row_elements, columns):
    data = []
    for row in row_elements:
        row_data = {}
        for col in row.findall('{urn:schemas-microsoft-com:xml-analysis:rowset}*'):
            column_name = col.tag.split('}')[1]
            column_value = col.text
            if column_name in columns:  # Check if the column is in the columns dictionary
                row_data[columns[column_name]] = column_value if column_value is not None else np.nan  # Insert NaN if value is missing
        data.append(row_data)
    return data

#pulling the analytics report and extracting data
def get_report(response):
    # Parse the XML response
    root = ET.fromstring(response.content)
    # Extract the ResumptionToken and IsFinished elements from the response XML
    resumption_token_element = root.find('.//ResumptionToken')
    resumption_token = resumption_token_element.text if resumption_token_element is not None else None
    is_finished_element = root.find('.//IsFinished')
    if is_finished_element is not None:
        is_finished = is_finished_element.text
    else:
        is_finished = None

    namespaces = {
        'xmlns': 'urn:schemas-microsoft-com:xml-analysis:rowset',
        'xsd': 'http://www.w3.org/2001/XMLSchema',
        'saw-sql': 'urn:saw-sql'
    }
    columns = {}

    for i, col in enumerate(root.findall('.//xsd:element[@saw-sql:columnHeading]', namespaces)):
        columns[f'Column{i}'] = col.get('{urn:saw-sql}columnHeading', col.get('name'))
    
    # Extract the data from the XML and store it in a list of dictionaries
    all_data = []
    data = extract_data(root.findall('.//{urn:schemas-microsoft-com:xml-analysis:rowset}Row'), columns)
    all_data.extend(data)
    
    # Iterate through subsequent GET requests until IsFinished is True
    while is_finished == 'false':
        # Make the next GET request with the ResumptionToken
        response = requests.get(token_path + resumption_token)

        # Extract the ResumptionToken and IsFinished elements from the response XML
        root = ET.fromstring(response.content)
        is_finished = root.find('.//IsFinished').text

        # Process the response data
        data = extract_data(root.findall('.//{urn:schemas-microsoft-com:xml-analysis:rowset}Row'), columns)
        all_data.extend(data)

    report_df = pd.DataFrame(all_data)

    # Drop the first column of the dataframe
    if not report_df.empty:
        report_df = report_df.drop(report_df.columns[0], axis=1)

    # Reset the index of the dataframe
    report_df = report_df.reset_index(drop=True)

    return report_df


def get_and_concat_report(report_path, report_data_df):
    response = requests.get(report_path)
    report_df = get_report(response)
    print(len(report_df))
    return pd.concat([report_data_df, report_df], ignore_index=True)

# Define a custom function to concatenate non-NaN values and deduplicate the notes fields which contain information about 
# online access type for each ebook
def concatenate_and_deduplicate_notes(row):
    notes = [str(row['Portfolio Public Note']), str(row['Portfolio Authentication Note']),
             str(row['Service Authentication Note']), str(row['Electronic location and access'])]
    non_nan_notes = set(note for note in notes if note != 'nan')
    return '; '.join(non_nan_notes)


## 2. Set the date 

In [4]:
#set the date string for the rest of this notebook
date_str = '2026-06-09'

## 3. Read in the searchresults file from the bookstore
## make sure to point read_excel to the correct file location!

In [5]:
#load most recent file from bookstore
# you will need to have saved the automated bookstore file using format "searchresults_YYYY-MM-DD.xlsx" to your computer
df = pd.read_excel(r'searchresults_'+date_str+'.xlsx')

# Preview the sorted DataFrame
df


,Academic Term,Academic Department Name,Section Code,Instructor Name,ISBN,Short Title,Book Author,Book Publisher,Enrollment Capacity
0,Summer Mini 2026,ACCOUNTING,202625-ACCT.200.001-70001,"Jordan, Latonya",9781264453948,CONNECT ONLINE ACCESS FOR SURVEY OF ACCOUNTING...,EDMONDS,MCGRAW-HILL HIGHER EDUCATION (US),100
1,Summer Mini 2026,ADVERTISING,202625-ADVT.480.001-70234,"Najera, Christina",9781483315430,CONTROVERSIES IN CONTEMPORARY ADVERTISING,"SHEEHAN, KIM BARTEL AND SHEEHAN, KIM B.","SAGE PUBLICATIONS, INCORPORATED",25
2,Summer Mini 2026,BIOCHEMISTRY CELLULAR MOLECULAR BIOLOGY,202625-BCMB.419.001-70259,"Korotych, Olena",9780470471319,FUNDAMENTAL LABORATORY APPROACHES FOR BIOCHEMI...,"NINFA, ALEXANDER J., BENORE, MARILEE, AND BALL...","WILEY & SONS, INCORPORATED, JOHN",12
3,Summer Mini 2026,BUSINESS ADMINISTRATION,202625-BUAD.200.001-70204,"Grant, Melissa",9781737565307,CAREER AI PLATFORM-QUINNCIA 2 YEAR ACCESS,QUINNCIA,QUINNCIA,50
4,Summer Mini 2026,COMMUNICATION STUDIES,202625-CMST.210.002-70420,"Lively, Traci",9781533943767,CUSTOM COURSE MATERIAL F/CMST 210 W/ACHIEVE,HAAS,MCMILLAN,20
...,...,...,...,...,...,...,...,...,...
890,Summer 2026,THEATRE,202630-THEA.100.392-82104,"Russell, Neno",9798214354422,THE ART OF THEATRE: THEN AND NOW,DOWNS,CENGAGE,30
891,Summer 2026,THEATRE,202630-THEA.100.501-83315,"Russell, Neno",9798214354422,THE ART OF THEATRE: THEN AND NOW,DOWNS,CENGAGE,32
892,Summer 2026,"WOMEN, GENDER AND SEXUALITY",202630-WGS.200.301-87025,"Kendrick, Samantha",9781101992678,THE PORTABLE FEMINIST READER,NaN,PENGUIN PUBLISHING GROUP,20
893,Summer 2026,"WOMEN, GENDER AND SEXUALITY",202630-WGS.200.302-87026,"Kendrick, Samantha",9781101992678,THE PORTABLE FEMINIST READER,NaN,PENGUIN PUBLISHING GROUP,5


## 4. do some work on the bookstore-provided file to generate a list of ISBNs that 
## we will use in a later step

In [6]:
#create list of ISBNs for use in Alma Set creation
#do some pre-processing on the data for cleanup purposes
isbns = df[['ISBN']]
isbns = isbns.dropna()
isbns = isbns.drop_duplicates()
isbns['ISBN'] = isbns['ISBN'].apply(lambda x: x.replace('-',''))
isbns = isbns.astype(str)
isbns = isbns[(isbns['ISBN'].str.len() == 13) & (isbns['ISBN'].str.isdigit())]
isbns = isbns.reset_index()
isbns = isbns.drop('index', axis = 1)
isbns = isbns.iloc[isbns['ISBN'].str.len().sort_values(ascending = False).index]
isbns

,ISBN
0,9781264453948
289,9781071962893
263,9780323846554
262,9780826182142
261,9781948057882
...,...
126,9798214153506
125,9781778771606
124,9780137319473
123,9781903544877


In [7]:
isbns.dtypes

ISBN    object
dtype: object

## Fix Known Mismatched ISBNs in the cell below.

These titles can be possibly identified with the fuzzy title matching section (Section #7 later in this code), or by checking their Alma holding and VolBooks title information against each other manually.

In [8]:
#Fix known mismatching ISBNs

for i in range(isbns.shape[0]):
    if isbns.at[i, 'ISBN'] == '9780191037627': #Silas Marner
        print('Found Silas Marner')
        isbns.at[i, 'ISBN'] = '9780191037610'    
    if isbns.at[i, 'ISBN'] == '9781040675892': #Sick Enough
        print('Found Sick Enough')       
        isbns.at[i, 'ISBN'] = '9781041036487'   
    if isbns.at[i, 'ISBN'] == '9781440855375': #Evaluation and Measurement of Library Services
        print('Found Evaluation and Measurement of Library Services')     
        isbns.at[i, 'ISBN'] = '9781440855368'     
    if isbns.at[i, 'ISBN'] == '9781003155119': #Information Science: The Basics
        print('Found Information Science')     
        isbns.at[i, 'ISBN'] = '9780367725204'      
    if isbns.at[i, 'ISBN'] == '9781119628149': #International Financial Statement Analysis
        print('Found International Financial Statement Analysis')    
        isbns.at[i, 'ISBN'] = '9781119682141'     
    if isbns.at[i, 'ISBN'] == '9781351133135': #Counseling Children and Adolescents
        print('Found Counseling Children and Adolescents')  
        isbns.at[i, 'ISBN'] = '9781351133159'   
    if isbns.at[i, 'ISBN'] == '9781501505935': #Demand and Supply Integration
        print('Found Demand and Supply Integration')  
        isbns.at[i, 'ISBN'] = '9781501506024'  
    if isbns.at[i, 'ISBN'] == '9781506352916': #Performance Appraisal and Management
        print('Found Performance Appraisal and Management')  
        isbns.at[i, 'ISBN'] = '9781506352886'  
    if isbns.at[i, 'ISBN'] == '9781317346005': #On the History of Political Philosophy
        print('Found On the History of Political Philosophy')
        isbns.at[i, 'ISBN'] = '9781317346012'
    if isbns.at[i, 'ISBN'] == '9781000690835': #Child-Centered Play Therapy
        print('Found Child-Centered Play Therapy')
        isbns.at[i, 'ISBN'] = '9781003260431' 
    if isbns.at[i, 'ISBN'] == '9781483315430': #Controversies in Contemporary Advertising
        print('Found Controversies in Contemporary Advertising')
        isbns.at[i, 'ISBN'] = '9781452233130' 
    if isbns.at[i, 'ISBN'] == '9781538139332': #Cooperation and Conflict Between State and Local Government
        print('Found Cooperation and Conflict Between State and Local Government')
        isbns.at[i, 'ISBN'] = '9798216405665' 
    if isbns.at[i, 'ISBN'] == '9781315218489': #Advanced Linear Algebra for Engineers with MATLAB
        print('Found Advanced Linear Algebra for Engineers with MATLAB') 
        isbns.at[i, 'ISBN'] = '9781420095241' 
    if isbns.at[i, 'ISBN'] == '9781351834377': #Advanced Linear Algebra for Engineers with MATLAB, second ISBN
        print('Found Advanced Linear Algebra for Engineers with MATLAB')
        isbns.at[i, 'ISBN'] = '9781420095241'  
    if isbns.at[i, 'ISBN'] == '9780393069754': #Beowulf
        print('Found Beowulf') 
        isbns.at[i, 'ISBN'] = '9781350212718'
    if isbns.at[i, 'ISBN'] == '9781483322100': #Conflict, War, and Peace: An Introduction to Scientific Research
        print('Found Conflict, War, and Peace')
        isbns.at[i, 'ISBN'] = '9781071934111'
    if isbns.at[i, 'ISBN'] == '9780742575622': #Criminologiccal Theory: An Analysis of its Underlying Assumptions
        print('Found Criminological Theory')
        isbns.at[i, 'ISBN'] = '9798216404972'
    if isbns.at[i, 'ISBN'] == '9780191007774': #Global Environments Through the Quarternary: Exploring Environmental Change
        print('Found Global Environments Through the Quarternary')
        isbns.at[i, 'ISBN'] = '9780191810169'
    if isbns.at[i, 'ISBN'] == '9780191606373': #History: A Very Short Introduction
        print('Found History')
        isbns.at[i, 'ISBN'] = '9780191540189'
    if isbns.at[i, 'ISBN'] == '9780124079489': #Introduction to Probability Models
        print('Found Introduction to Probability Models')
        isbns.at[i, 'ISBN'] = '9780125984560'
    if isbns.at[i, 'ISBN'] == '9781948057882': #John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
        print("Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition")
        print('Make sure this is the Unlimited version from ProQuest, not the 1 user from R2 Digital!')
        isbns.at[i, 'ISBN'] = '9781948057899'
    if isbns.at[i, 'ISBN'] == '9780838937952': #Metadata
        print('Found Metadata')
        isbns.at[i, 'ISBN'] = '9780838948637'
    if isbns.at[i, 'ISBN'] == '9780307757890': #Paradise Lost
        print('Found Paradise Lost')
        isbns.at[i, 'ISBN'] = '9781317865735'
    if isbns.at[i, 'ISBN'] == '9781466887787': #14-18: Understanding the Great War
        print('Found 14-18 Understanding the Great War')
        isbns.at[i, 'ISBN'] = '9780809046430'
    if isbns.at[i, 'ISBN'] == '9781400030156': #The Big Sleep
        print('Found The Big Sleep')
        isbns.at[i, 'ISBN'] = '9781479403776'
    if isbns.at[i, 'ISBN'] == '9780191629112': #The Book of Marvels and Travels
        print('Found The Book of Marvels and Travels')
        isbns.at[i, 'ISBN'] = '9780191629105'
    if isbns.at[i, 'ISBN'] == '9780140449617': #The Death of Ivan Ilyich and Other Stories
        print('Found The Death of Ivan Ilyich')
        isbns.at[i, 'ISBN'] = '9780191649042'
    if isbns.at[i, 'ISBN'] == '9781479823550': #Forced Out: Migrant Mothers in Search of Refuge and Hope
        print('Found Forced Out')
        isbns.at[i, 'ISBN'] = '9781479823529'
    if isbns.at[i, 'ISBN'] == '9780198745686': #Imperial Apocalypse: The Great War and the Destruction of the Russian Empire
        print('Found Imperial Apocalypse')
        isbns.at[i, 'ISBN'] = '9781644699522'
    if isbns.at[i, 'ISBN'] == '9780262367509': #Introduction to Algorithms, Fourth Edition
        print('Found Introduction to Algorithms')
        isbns.at[i, 'ISBN'] = '9780262046305'
    if isbns.at[i, 'ISBN'] == '9780813178288': #John Hervey Wheeler, Black Banking, and the Economic Struggle for Civil Rights]
        print('Found John Hervey Wheeler')
        isbns.at[i, 'ISBN'] = '9780813178271'
    if isbns.at[i, 'ISBN'] == '9781452289236': #Mental Health Case Management: A Practical Guide
        print('Found Mental Health Case Management')
        isbns.at[i, 'ISBN'] = '9781452256078'
    if isbns.at[i, 'ISBN'] == '9781101576984': #Merchants of Culture The Publishing Business in the Twenty-First Century
        print('Found Merchants of Culture')
        isbns.at[i, 'ISBN'] = '9780745661421'
    if isbns.at[i, 'ISBN'] == '9781319049966': #Narrative of the Life of Frederick Douglass: An American Slave, Written by Himself
        print('Found Frederick Douglass')
        isbns.at[i, 'ISBN'] = '9780760355466'
    if isbns.at[i, 'ISBN'] == '9780199720149': #The Oxford Handbook of Engineering and Technology in the Classical World
        print('Found Handbook of Engineering and Technology')
        isbns.at[i, 'ISBN'] = '9780195187311'
    if isbns.at[i, 'ISBN'] == '9781350113008': #Performance Drawing
        print('Found Performance Drawing')
        isbns.at[i, 'ISBN'] = '9781350113015'
    if isbns.at[i, 'ISBN'] == '9780191624711': #Persuasion
        print('Found Persuasion')
        isbns.at[i, 'ISBN'] = '9781316848630'
    if isbns.at[i, 'ISBN'] == '9781009314848': #The Gas Mask in Interwar Germany: Visions of Chemical Modernity
        print('Found Gas Mask')
        isbns.at[i, 'ISBN'] = '9781009314862'
    if isbns.at[i, 'ISBN'] == '9780385093309': #Good-Bye to All That
        print('Found Good-Bye to All That')
        isbns.at[i, 'ISBN'] = '9781774649367'
    if isbns.at[i, 'ISBN'] == '9781119837909': #Research Ethics for Scientists
        print('Found Research Ethics for Scientists')
        isbns.at[i, 'ISBN'] = '9781119978862'
    if isbns.at[i, 'ISBN'] == '9780307432568': #The Return of the Soldier
        print('Found The Return of the Soldier')
        isbns.at[i, 'ISBN'] = '9780486846545'
    if isbns.at[i, 'ISBN'] == '9781496203168': #Siberian Exile
        print('Found Siberian Exile')
        isbns.at[i, 'ISBN'] = '9780803299597'
    if isbns.at[i, 'ISBN'] == '9780190879976': #Teaching Music Theory
        print('Found Teaching Music Theory')
        isbns.at[i, 'ISBN'] = '9780197510575'
    if isbns.at[i, 'ISBN'] == '9780197644058': #An Uneasy Peace
        print('Found An Uneasy Peace')
        isbns.at[i, 'ISBN'] = '9780197619407'

Found Controversies in Contemporary Advertising
Found Child-Centered Play Therapy
Found Beowulf
Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
Make sure this is the Unlimited version from ProQuest, not the 1 user from R2 Digital!
Found On the History of Political Philosophy
Found Demand and Supply Integration


## 5. Save the isbns to an Excel file

In [9]:
#create new itemized set in Alma using this isbn file:
isbns.to_excel(date_str+'_isbns.xlsx', index = False)

# after the set is created, export the results (All Fields) and save the file using format "YYYY-MM-DD_set_output.xlsx"

## Use the isbn file to create an itemized set in alma
## After the set is created, export the results (All Fields) and save using the file name format "YYYY-MM-DD_set_output.xlsx" to the same location that all your other files for this process live in

## 6. The following cell will use the set_output you just made to call on the alma analytics API, looking for any matching ISBNs

In [5]:
#use the alma API and the set results to get all ISBNs for any MMS IDs found in set
#analytics report: Collections folder > "Textbook ISBNs for Overlap Report_TM"

#create list of mms_ids from set output
mms_ids = pd.read_excel(date_str+"_set_output.xlsx")
#mms_ids = pd.read_excel('ProQuest to EBSCO Check 2-4.xlsx')
mms_ids = mms_ids[['MMS ID']]

# Get the list of MMS IDs from the DataFrame
mms_list = mms_ids['MMS ID'].tolist()

# Define the batch size for API call(s)
batch_size = 50

# Create a list to store the batches
batches = []

# Iterate through the MMS ID list and create batches
for i in range(0, len(mms_list), batch_size):
    batch = mms_list[i : i + batch_size]
    batches.append(batch)

#create an empty list into which we will put api call results
df_list = []

#for each batch of mms ids, call analytics api
for batch in batches:
    values_xml = '\n'.join(f'<sawx:expr xsi:type="xsd:string">{value}</sawx:expr>' for value in batch)
    formatted_xml_snippet = xml_snippet.format(values_xml)

    url_encoded_filter_query = urllib.parse.quote(formatted_xml_snippet)

    full_path = base_path + url_encoded_path + r'&filter=' + url_encoded_filter_query

    report_data_df = get_and_concat_report(full_path, pd.DataFrame())
    df_list.append(report_data_df)
    
# concatenate together all the api call results into one big dataframe
combined_df = pd.concat(df_list, ignore_index=True)
combined_df


39
19


,Electronic location and access,ISBN (Normalized),ISBN,MMS Id,Title,Electronic Collection Public Name,Availability,Portfolio Id,Service Authentication Note,No. of Available Portfolio,Portfolio Public Note,Portfolio Authentication Note
0,https://www.taylorfrancis.com/books/9781003260...,1003260438; 9781003260431; 9781032196879; 1032...,1003260438; 9781003260431; 9781032196879; 1032...,9926275286002311,Child-centered play therapy : a practical guid...,Taylor & Francis Ebooks (EBA),Available,53390423470002311,Online Access: Unlimited Users,0,NaN,NaN
1,https://www.taylorfrancis.com/books/9781003260...,1003260438; 9781003260431; 9781032196879; 1032...,1003260438; 9781003260431; 9781032196879; 1032...,9926275286002311,Child-centered play therapy : a practical guid...,Taylor & Francis Ebooks EBA Purchased,Available,53407973790002311,NaN,0,Online Access- Unlimited users (DRM-Free),NaN
2,https://www.taylorfrancis.com/books/9781003524090,1003524095; 9781003524090; 1040524826; 9781040...,1003524095; 9781003524090; 1040524826; 9781040...,9926907593802311,An evidence-based guide to assistive technolog...,Taylor & Francis Ebooks (EBA),Available,53452318750002311,Online Access: Unlimited Users,1,NaN,NaN
3,https://link.springer.com/978-3-031-05740-3 Un...,9783031057403; 3031057406; 9783031057397; 3031...,9783031057403; 3031057406; 9783031057397; 3031...,9926837078602311,Applied Artificial Intelligence in Business : ...,Springer Ebooks Individually Purchased,Available,53446362200002311,NaN,1,Unlimited Access,NaN
4,https://doi.org/10.5040/9781350212732?locatt=l...,1350212717; 9781350212718; 1350212733; 9781350...,1350212717; 9781350212718; 1350212733; 9781350...,9926582988102311,"Beowulf : poem, poet and hero /",Bloomsbury Ebooks Individually Purchased,Available,53422236940002311,NaN,1,NaN,Online Access - Unlimited users (DRM-free)
5,https://www.taylorfrancis.com/books/9781003260...,1003260438; 9781003260431; 9781032196879; 1032...,1003260438; 9781003260431; 9781032196879; 1032...,9926275286002311,Child-centered play therapy : a practical guid...,Taylor & Francis Ebooks EBA Purchased,Available,53407996680002311,NaN,1,Online Access- Unlimited users (DRM-Free),NaN
6,https://sk.sagepub.com/book/mono/controversies...,9781452233130; 1452233136; 9780761926351; 0761...,9781452233130; 1452233136; 9780761926351; 0761...,9926908933302311,Controversies in contemporary advertising,SAGE Research Methods Books & Reference,Available,53452449920002311,NaN,1,Unlimited Access,NaN
7,https://www.taylorfrancis.com/books/9780203107...,0415887518; 9780415887519; 0415643465; 9780415...,0415887518; 9780415887519; 0415643465; 9780415...,9926414501702311,Core Competencies in Cognitive-Behavioral Ther...,Taylor & Francis Ebooks EBA Purchased,Available,53407972830002311,NaN,1,Online Access- Unlimited users (DRM-Free),NaN
8,https://www.taylorfrancis.com/books/9781003278290,1000566269; 9781000566260; 1003278299; 9781003...,1000566269; 9781000566260; 1003278299; 9781003...,9926211119402311,Ethics of data and analytics : concepts and ca...,Taylor & Francis Ebooks (EBA),Available,53384931230002311,Online Access: Unlimited Users,1,NaN,NaN
9,https://search.ebscohost.com/login.aspx?direct...,0826182143; 9780826182142; 9780826182135; 0826...,0826182143; 9780826182142; 9780826182135; 0826...,9926521386802311,Evidence-based practice improvement : merging ...,Ebooks moved from ProQuest to EBSCO,Available,53450840880002311,NaN,1,NaN,Online Access - 3 Users


In [7]:
combined_df.to_excel('Alma Textbook List Check Summer 2026.xlsx')

In [12]:
combined_df.sort_values(by = 'Title')['Title'].unique()

array(['An evidence-based guide to assistive technology : meeting the needs of all students with disabilities /',
       'Applied Artificial Intelligence in Business : Concepts and Cases.',
       'Beowulf : poem, poet and hero /',
       'Child-centered play therapy : a practical guide to therapeutic relationships with children /',
       'Controversies in contemporary advertising',
       'Core Competencies in Cognitive-Behavioral Therapy : Becoming a Highly Effective and Competent Cognitive-Behavioral Therapist /',
       'Deaf culture : exploring deaf communities in the United States /',
       'Demand and Supply Integration : The Key to World-Class Demand Forecasting, Second Edition.',
       'Ethics of data and analytics : concepts and cases /',
       'Evicted poverty and profit in the American city',
       'Evidence-based practice improvement : merging evidence-based practice and quality improvement /',
       'Global writing for public relations : connecting in English with s

## 7. Check for mismatched ISBNs using Fuzzy Matching

Since titles often have several different ISBNs, and we only receive one ISBN from VolBooks, we need to make sure as many of those ISBNs match up as possible. The cell below checks for any of these potential mismatches.

### If a title's ISBN does not align between VolBooks and Alma, go to the ISBN fixing cell in Section #4 and edit the VolBooks holdings to match the Alma holdings.

**NOTE:** This code finds the closest matching title within the VolBooks and Alma lists, and thus it may not always be a direct match. Double-check the code before changing any ISBNs.

In [13]:
## Load the Alma Data

volBooksTitleList = df['Short Title'].unique()
almaUnlimited = pd.read_excel('Textbook ISBN Checking File May 4.xlsx')

FileNotFoundError: [Errno 2] No such file or directory: 'Textbook ISBN Checking File May 4.xlsx'

In [ ]:
## Alma ISBN Splitting Code
combinedCopy = almaUnlimited

combinedCopy['ISBN (Normalized) List'] = combinedCopy['ISBN (Normalized)'].apply(lambda x: str(x).split(";"))

expandedCopy = combinedCopy.explode('ISBN (Normalized) List')

expandedCopy['ISBN (Normalized) List'] = expandedCopy['ISBN (Normalized) List'].str.strip()

In [ ]:
## Clean and Tokenize the Titles

def TextPreProcess(title):
    title = str(title)
    title = title.lower()
    title = title.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(title)

    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

## Load Title Lists
df['Tokenized Title'] = df['Short Title'].apply(TextPreProcess)
expandedCopy['Tokenized Title'] = expandedCopy['Title'].apply(TextPreProcess)

In [ ]:
## Create Stems and Rejoin Titles

stemmer = SnowballStemmer(language = 'english')

def StemTokens(tokens):
    stems = [stemmer.stem(token) for token in tokens]
    return ' '.join(stems)

df['Stemmed Title'] = df['Tokenized Title'].apply(StemTokens)
expandedCopy['Stemmed Title'] = expandedCopy['Tokenized Title'].apply(StemTokens)

In [ ]:
## Initialize Information

volBooksTitleList = df['Stemmed Title'].unique()
almaTitleList = expandedCopy['Stemmed Title'].unique()

almaTitle = pd.Series([])
volBooksTitle = pd.Series([])
almaISBN = pd.Series([])
volBooksISBN = pd.Series([])
almaStem = pd.Series([])
volBooksStem = pd.Series([])

In [ ]:
## Find the Potential Titles with Mismatched ISBNs

for i in range(len(volBooksTitleList)):
    titleName = volBooksTitleList[i]
    bestResult = process.extractOne(titleName, almaTitleList, scorer = fuzz.token_sort_ratio)
    
    selectVolBooksRows = df[df['Stemmed Title'] == titleName]
    volBooksStem = pd.concat([volBooksStem, pd.Series([selectVolBooksRows['Stemmed Title'].iloc[0]])], ignore_index = True)
    volBooksISBN = pd.concat([volBooksISBN, pd.Series([selectVolBooksRows['ISBN'].iloc[0]])], ignore_index = True)
    volBooksTitle = pd.concat([volBooksTitle, pd.Series([selectVolBooksRows['Short Title'].iloc[0]])], ignore_index = True)
    
    selectAlmaRows = expandedCopy[expandedCopy['Stemmed Title'] == bestResult[0]]
    almaStem = pd.concat([almaStem, pd.Series([selectAlmaRows['Stemmed Title'].iloc[0]])], ignore_index = True)
    almaISBN = pd.concat([almaISBN, pd.Series([selectAlmaRows['ISBN (Normalized) List'].iloc[0]])], ignore_index = True)
    almaTitle = pd.concat([almaTitle, pd.Series([selectAlmaRows['Title'].iloc[0]])], ignore_index = True)
    
    #for j in range(expandedCopy.shape[0]):
    #    if expandedCopy['Title'].iloc[j] == bestResult:
    #        almaISBN = pd.concat(volBooksISBN, pd.Series([expandedCopy.at[j, 'ISBN (Normalized) List']]), ignore_index = True)
    #        almaTitle = pd.concat(volBooksTitle, pd.Series([bestResult]), ignore_index = True)
    #for k in range(df.shape[0]):
    #    if expandedCopy['Short Title'].iloc[k] == titleName:
    #        volBooksISBN = pd.concat(almaISBN, pd.Series([df.at[k, 'ISBN']]), ignore_index = True)
    #        volBooksTitle = pd.concat(almaTitle, pd.Series([titleName]), ignore_index = True)

titleComparisonTable = pd.concat([volBooksStem, almaStem, volBooksISBN, almaISBN, volBooksTitle, almaTitle], axis = 1)
    
titleComparisonTable.columns = ['VolBooks Stemmed Title', 'Alma Stemmed Title', 'VolBooks ISBN', 'Alma ISBN', 'VolBooks Raw Title', 'Alma Raw Title']

titleComparisonTable

In [ ]:
titleComparisonTable.to_excel(date_str+'ISBN Matching Table.xlsx', index = False)

## 8. Continue cleaning up the VolBooks report

Return here once all ISBNs have been fixed above.

In [6]:
combined_df['Service Authentication Note']

0     Online Access: Unlimited Users
1                                NaN
2     Online Access: Unlimited Users
3                                NaN
4                                NaN
5                                NaN
6                                NaN
7                                NaN
8     Online Access: Unlimited Users
9                                NaN
10                               NaN
11                               NaN
12                               NaN
13                               NaN
14                               NaN
15                               NaN
16                               NaN
17                               NaN
18                               NaN
19                               NaN
20                               NaN
21                               NaN
22                               NaN
23                               NaN
24                               NaN
25                               NaN
26                               NaN
2

In [14]:
combined_df['Electronic location split'] = combined_df['Electronic location and access'].apply(lambda x: x.split() if isinstance(x, str) else x)

combined_df['Electronic location'] = combined_df['Electronic location split'].apply(lambda x: ''.join(map(str, x[:1])) if isinstance(x, list) else x)

# Extract items excluding the first item from each list
combined_df['Electronic access'] = combined_df['Electronic location split'].apply(lambda x: ' '.join(x[1:]) if isinstance(x, list) else x)


In [15]:
combined_df['Electronic access']

0      Online Access- Unlimited users (DRM-Free)
1      Online Access- Unlimited users (DRM-Free)
2                                               
3              Unlimited Access Unlimited Access
4     Online Access - Unlimited users (DRM-free)
5      Online Access- Unlimited users (DRM-Free)
6              Unlimited Access Unlimited Access
7      Online Access- Unlimited users (DRM-Free)
8                                               
9                                               
10                                           NaN
11                                           NaN
12             Unlimited Access Unlimited Access
13             Unlimited Access Unlimited Access
14                                              
15             Unlimited Access Unlimited Access
16             Unlimited Access Unlimited Access
17                Click to access digital title.
18     Online Access- Unlimited users (DRM-Free)
19                                 Click to View
20             Unlim

## 9. save the alma analytics api output as an excel file

In [16]:
#save the big, concatenated dataframe to disk
combined_df.to_excel(date_str+'_analytics_output.xlsx', index = False)

## do some work on the dataframe

In [17]:
# Create a new column 'Concatenated Notes' with deduplication
combined_df['Concatenated Notes'] = combined_df.apply(concatenate_and_deduplicate_notes, axis=1)

#filter to only show rows where the No. of Available Portfolios is not zero
combined_df = combined_df.loc[combined_df['No. of Available Portfolio'] != 0]


# Display the updated DataFrame
combined_df


,Electronic location and access,ISBN (Normalized),ISBN,MMS Id,Title,Electronic Collection Public Name,Availability,Portfolio Id,Service Authentication Note,No. of Available Portfolio,Portfolio Public Note,Portfolio Authentication Note,Electronic location split,Electronic location,Electronic access,Concatenated Notes
0,https://www.taylorfrancis.com/books/9781003260...,1003260438; 9781003260431; 9781032196879; 1032...,1003260438; 9781003260431; 9781032196879; 1032...,9926275286002311,Child-centered play therapy : a practical guid...,Taylor & Francis Ebooks (EBA),Available,53390423470002311,Online Access: Unlimited Users,0,NaN,NaN,[https://www.taylorfrancis.com/books/978100326...,https://www.taylorfrancis.com/books/9781003260431,Online Access- Unlimited users (DRM-Free),Online Access: Unlimited Users; https://www.ta...
1,https://www.taylorfrancis.com/books/9781003260...,1003260438; 9781003260431; 9781032196879; 1032...,1003260438; 9781003260431; 9781032196879; 1032...,9926275286002311,Child-centered play therapy : a practical guid...,Taylor & Francis Ebooks EBA Purchased,Available,53407973790002311,NaN,0,Online Access- Unlimited users (DRM-Free),NaN,[https://www.taylorfrancis.com/books/978100326...,https://www.taylorfrancis.com/books/9781003260431,Online Access- Unlimited users (DRM-Free),Online Access- Unlimited users (DRM-Free); htt...
2,https://www.taylorfrancis.com/books/9781003524090,1003524095; 9781003524090; 1040524826; 9781040...,1003524095; 9781003524090; 1040524826; 9781040...,9926907593802311,An evidence-based guide to assistive technolog...,Taylor & Francis Ebooks (EBA),Available,53452318750002311,Online Access: Unlimited Users,1,NaN,NaN,[https://www.taylorfrancis.com/books/978100352...,https://www.taylorfrancis.com/books/9781003524090,,Online Access: Unlimited Users; https://www.ta...
3,https://link.springer.com/978-3-031-05740-3 Un...,9783031057403; 3031057406; 9783031057397; 3031...,9783031057403; 3031057406; 9783031057397; 3031...,9926837078602311,Applied Artificial Intelligence in Business : ...,Springer Ebooks Individually Purchased,Available,53446362200002311,NaN,1,Unlimited Access,NaN,"[https://link.springer.com/978-3-031-05740-3, ...",https://link.springer.com/978-3-031-05740-3,Unlimited Access Unlimited Access,https://link.springer.com/978-3-031-05740-3 Un...
4,https://doi.org/10.5040/9781350212732?locatt=l...,1350212717; 9781350212718; 1350212733; 9781350...,1350212717; 9781350212718; 1350212733; 9781350...,9926582988102311,"Beowulf : poem, poet and hero /",Bloomsbury Ebooks Individually Purchased,Available,53422236940002311,NaN,1,NaN,Online Access - Unlimited users (DRM-free),[https://doi.org/10.5040/9781350212732?locatt=...,https://doi.org/10.5040/9781350212732?locatt=l...,Online Access - Unlimited users (DRM-free),Online Access - Unlimited users (DRM-free); ht...
5,https://www.taylorfrancis.com/books/9781003260...,1003260438; 9781003260431; 9781032196879; 1032...,1003260438; 9781003260431; 9781032196879; 1032...,9926275286002311,Child-centered play therapy : a practical guid...,Taylor & Francis Ebooks EBA Purchased,Available,53407996680002311,NaN,1,Online Access- Unlimited users (DRM-Free),NaN,[https://www.taylorfrancis.com/books/978100326...,https://www.taylorfrancis.com/books/9781003260431,Online Access- Unlimited users (DRM-Free),Online Access- Unlimited users (DRM-Free); htt...
6,https://sk.sagepub.com/book/mono/controversies...,9781452233130; 1452233136; 9780761926351; 0761...,9781452233130; 1452233136; 9780761926351; 0761...,9926908933302311,Controversies in contemporary advertising,SAGE Research Methods Books & Reference,Available,53452449920002311,NaN,1,Unlimited Access,NaN,[https://sk.sagepub.com/book/mono/controversie...,https://sk.sagepub.com/book/mono/controversies...,Unlimited Access Unlimited Access,https://sk.sagepub.com/book/mono/controversies...
7,https://www.taylorfrancis.com/books/9780203107...,0415887518; 9780415887519; 0415643465; 9780415...,0415887518; 9780415887519; 0415643465; 978

## 10. save the analytics_output file as csv

In [18]:
#save the dataframe to disk once again, only this time as a csv
combined_df.to_csv(date_str+'_analytics_output.csv', index = False)

## 11. Combine Data Files for Exporting

In [19]:
results = pd.read_excel(date_str+'_set_output.xlsx') #From Alma Directly
#results = pd.read_excel('ProQuest to EBSCO Check 2-4.xlsx')
bookstoreList = pd.read_excel('searchresults_'+date_str+'.xlsx') #From VolBooks Directly
textbookISBNs = combined_df #Alma Cleaned-Up Version

## Make sure that any ISBNs that were fixed before loading into Alma are also fixed here!

In [20]:
#Fix known mismatching ISBNs in bookstoreList

for i in range(bookstoreList.shape[0]):
    if bookstoreList.at[i, 'ISBN'] == '9780191037627': #Silas Marner
        print('Found Silas Marner')
        bookstoreList.at[i, 'ISBN'] = '9780191037610'    
    if bookstoreList.at[i, 'ISBN'] == '9781040675892': #Sick Enough
        print('Found Sick Enough')       
        bookstoreList.at[i, 'ISBN'] = '9781041036487'   
    if bookstoreList.at[i, 'ISBN'] == '9781440855375': #Evaluation and Measurement of Library Services
        print('Found Evaluation and Measurement of Library Services')     
        bookstoreList.at[i, 'ISBN'] = '9781440855368'     
    if bookstoreList.at[i, 'ISBN'] == '9781003155119': #Information Science: The Basics
        print('Found Information Science')     
        bookstoreList.at[i, 'ISBN'] = '9780367725204'      
    if bookstoreList.at[i, 'ISBN'] == '9781119628149': #International Financial Statement Analysis
        print('Found International Financial Statement Analysis')    
        bookstoreList.at[i, 'ISBN'] = '9781119682141'     
    if bookstoreList.at[i, 'ISBN'] == '9781351133135': #Counseling Children and Adolescents
        print('Found Counseling Children and Adolescents')  
        bookstoreList.at[i, 'ISBN'] = '9781351133159'   
    if bookstoreList.at[i, 'ISBN'] == '9781501505935': #Demand and Supply Integration
        print('Found Demand and Supply Integration')  
        bookstoreList.at[i, 'ISBN'] = '9781501506024'  
    if bookstoreList.at[i, 'ISBN'] == '9781506352916': #Performance Appraisal and Management
        print('Found Performance Appraisal and Management')  
        bookstoreList.at[i, 'ISBN'] = '9781506352886'  
    if bookstoreList.at[i, 'ISBN'] == '9781317346005': #On the History of Political Philosophy
        print('Found On the History of Political Philosophy')
        bookstoreList.at[i, 'ISBN'] = '9781317346012'
    if bookstoreList.at[i, 'ISBN'] == '9781000690835': #Child-Centered Play Therapy
        print('Found Child-Centered Play Therapy')
        bookstoreList.at[i, 'ISBN'] = '9781003260431' 
    if bookstoreList.at[i, 'ISBN'] == '9781483315430': #Controversies in Contemporary Advertising
        print('Found Controversies in Contemporary Advertising')
        bookstoreList.at[i, 'ISBN'] = '9781452233130' 
    if bookstoreList.at[i, 'ISBN'] == '9781538139332': #Cooperation and Conflict Between State and Local Government
        print('Found Cooperation and Conflict Between State and Local Government')
        bookstoreList.at[i, 'ISBN'] = '9798216405665' 
    if bookstoreList.at[i, 'ISBN'] == '9781315218489': #Advanced Linear Algebra for Engineers with MATLAB
        print('Found Advanced Linear Algebra for Engineers with MATLAB') 
        bookstoreList.at[i, 'ISBN'] = '9781420095241' 
    if bookstoreList.at[i, 'ISBN'] == '9781351834377': #Advanced Linear Algebra for Engineers with MATLAB, second ISBN
        print('Found Advanced Linear Algebra for Engineers with MATLAB')
        bookstoreList.at[i, 'ISBN'] = '9781420095241'  
    if bookstoreList.at[i, 'ISBN'] == '9780393069754': #Beowulf
        print('Found Beowulf') 
        bookstoreList.at[i, 'ISBN'] = '9781350212718'
    if bookstoreList.at[i, 'ISBN'] == '9781483322100': #Conflict, War, and Peace: An Introduction to Scientific Research
        print('Found Conflict, War, and Peace')
        bookstoreList.at[i, 'ISBN'] = '9781071934111'
    if bookstoreList.at[i, 'ISBN'] == '9780742575622': #Criminologiccal Theory: An Analysis of its Underlying Assumptions
        print('Found Criminological Theory')
        bookstoreList.at[i, 'ISBN'] = '9798216404972'
    if bookstoreList.at[i, 'ISBN'] == '9780191007774': #Global Environments Through the Quarternary: Exploring Environmental Change
        print('Found Global Environments Through the Quarternary')
        bookstoreList.at[i, 'ISBN'] = '9780191810169'
    if bookstoreList.at[i, 'ISBN'] == '9780191606373': #History: A Very Short Introduction
        print('Found History')
        bookstoreList.at[i, 'ISBN'] = '9780191540189'
    if bookstoreList.at[i, 'ISBN'] == '9780124079489': #Introduction to Probability Models
        print('Found Introduction to Probability Models')
        bookstoreList.at[i, 'ISBN'] = '9780125984560'
    if bookstoreList.at[i, 'ISBN'] == '9781948057882': #John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
        print("Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition")
        bookstoreList.at[i, 'ISBN'] = '9781948057899'
    if bookstoreList.at[i, 'ISBN'] == '9780838937952': #Metadata
        print('Found Metadata')
        bookstoreList.at[i, 'ISBN'] = '9780838948637'
    if bookstoreList.at[i, 'ISBN'] == '9780307757890': #Paradise Lost
        print('Found Paradise Lost')
        bookstoreList.at[i, 'ISBN'] = '9781317865735'
    if bookstoreList.at[i, 'ISBN'] == '9781466887787': #14-18: Understanding the Great War
        print('Found 14-18 Understanding the Great War')
        bookstoreList.at[i, 'ISBN'] = '9780809046430'
    if bookstoreList.at[i, 'ISBN'] == '9781400030156': #The Big Sleep
        print('Found The Big Sleep')
        bookstoreList.at[i, 'ISBN'] = '9781479403776'
    if bookstoreList.at[i, 'ISBN'] == '9780191629112': #The Book of Marvels and Travels
        print('Found The Book of Marvels and Travels')
        bookstoreList.at[i, 'ISBN'] = '9780191629105'
    if bookstoreList.at[i, 'ISBN'] == '9780140449617': #The Death of Ivan Ilyich and Other Stories
        print('Found The Death of Ivan Ilyich')
        bookstoreList.at[i, 'ISBN'] = '9780191649042'
    if bookstoreList.at[i, 'ISBN'] == '9781479823550': #Forced Out: Migrant Mothers in Search of Refuge and Hope
        print('Found Forced Out')
        bookstoreList.at[i, 'ISBN'] = '9781479823529'
    if bookstoreList.at[i, 'ISBN'] == '9780198745686': #Imperial Apocalypse: The Great War and the Destruction of the Russian Empire
        print('Found Imperial Apocalypse')
        bookstoreList.at[i, 'ISBN'] = '9781644699522'
    if bookstoreList.at[i, 'ISBN'] == '9780262367509': #Introduction to Algorithms, Fourth Edition
        print('Found Introduction to Algorithms')
        bookstoreList.at[i, 'ISBN'] = '9780262046305'
    if bookstoreList.at[i, 'ISBN'] == '9780813178288': #John Hervey Wheeler, Black Banking, and the Economic Struggle for Civil Rights]
        print('Found John Hervey Wheeler')
        bookstoreList.at[i, 'ISBN'] = '9780813178271'
    if bookstoreList.at[i, 'ISBN'] == '9781452289236': #Mental Health Case Management: A Practical Guide
        print('Found Mental Health Case Management')
        bookstoreList.at[i, 'ISBN'] = '9781452256078'
    if bookstoreList.at[i, 'ISBN'] == '9781101576984': #Merchants of Culture The Publishing Business in the Twenty-First Century
        print('Found Merchants of Culture')
        bookstoreList.at[i, 'ISBN'] = '9780745661421'
    if bookstoreList.at[i, 'ISBN'] == '9781319049966': #Narrative of the Life of Frederick Douglass: An American Slave, Written by Himself
        print('Found Frederick Douglass')
        bookstoreList.at[i, 'ISBN'] = '9780760355466'
    if bookstoreList.at[i, 'ISBN'] == '9780199720149': #The Oxford Handbook of Engineering and Technology in the Classical World
        print('Found Handbook of Engineering and Technology')
        bookstoreList.at[i, 'ISBN'] = '9780195187311'
    if bookstoreList.at[i, 'ISBN'] == '9781350113008': #Performance Drawing
        print('Found Performance Drawing')
        bookstoreList.at[i, 'ISBN'] = '9781350113015'
    if bookstoreList.at[i, 'ISBN'] == '9780191624711': #Persuasion
        print('Found Persuasion')
        bookstoreList.at[i, 'ISBN'] = '9781316848630'
    if bookstoreList.at[i, 'ISBN'] == '9781009314848': #The Gas Mask in Interwar Germany: Visions of Chemical Modernity
        print('Found Gas Mask')
        bookstoreList.at[i, 'ISBN'] = '9781009314862'
    if bookstoreList.at[i, 'ISBN'] == '9780385093309': #Good-Bye to All That
        print('Found Good-Bye to All That')
        bookstoreList.at[i, 'ISBN'] = '9781774649367'
    if bookstoreList.at[i, 'ISBN'] == '9781119837909': #Research Ethics for Scientists
        print('Found Research Ethics for Scientists')
        bookstoreList.at[i, 'ISBN'] = '9781119978862'
    if bookstoreList.at[i, 'ISBN'] == '9780307432568': #The Return of the Soldier
        print('Found The Return of the Soldier')
        bookstoreList.at[i, 'ISBN'] = '9780486846545'
    if bookstoreList.at[i, 'ISBN'] == '9781496203168': #Siberian Exile
        print('Found Siberian Exile')
        bookstoreList.at[i, 'ISBN'] = '9780803299597'
    if bookstoreList.at[i, 'ISBN'] == '9780190879976': #Teaching Music Theory
        print('Found Teaching Music Theory')
        bookstoreList.at[i, 'ISBN'] = '9780197510575'
    if bookstoreList.at[i, 'ISBN'] == '9780197644058': #An Uneasy Peace
        print('Found An Uneasy Peace')
        bookstoreList.at[i, 'ISBN'] = '9780197619407'

Found Controversies in Contemporary Advertising
Found Child-Centered Play Therapy
Found Beowulf
Found Beowulf
Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professiona

In [21]:
#pd.set_option('display.max_rows', None)
results.sort_values(by = 'Title')['Title'].unique()

array(['An evidence-based guide to assistive technology : meeting the needs of all students with disabilities / edited by Alice L. Rhodes and Victoria Slocum.',
       'Applied Artificial Intelligence in Business : Concepts and Cases.',
       "Beowulf : poem, poet and hero / Heather O'Donoghue.",
       'Child-centered play therapy : a practical guide to therapeutic relationships with children / Nancy H. Cochran, William J. Nordling and Jeff L. Cochran.',
       'Child-centered play therapy : a practical guide to therapeutic relationships with children / Nancy H. Cochran, William J. Nordling, Jeff L. Cochran.',
       'Controversies in contemporary advertising [electronic resource] / Kim Bartel Sheehan.',
       'Core Competencies in Cognitive-Behavioral Therapy : Becoming a Highly Effective and Competent Cognitive-Behavioral Therapist / Cory F. Newman.',
       'Deaf culture : exploring deaf communities in the United States / Irene W. Leigh, PhD, Jean F. Andrews, PhD, Raychelle L. Ha

In [22]:
textbookISBNs = textbookISBNs.astype({'MMS Id': 'int64'})



resultsPlusList = pd.merge(results, textbookISBNs, left_on = 'MMS ID', right_on = 'MMS Id', how = 'inner')

In [23]:
resultsPlusList

,Title_x,Alternate Graphic Representation,Type,Creator / Imprint,Subject,Series,Relation,Modification Date,Edition,Uniform Title,Medium Type,Language,Creation Date,Language of Cataloging,ISBN_x,ISBN (13),ISSN,Cluster ISSN,Record Number,MMS ID,Bibliographic Rank,Local Call Number,Other Classification Number,NLM Call Number,Dewey Decimal Classification Number,LC Call Number,Classification Part,UDC Classification Number,NAL Call Number,Handle Identifier,Peer Reviewed,Open Access,Physical Availability,Digital Availability,Electronic Availability,Collection Name,Collection ID,Orders,Requests,Physical Description Extent,Local Labels,Related Records,Courses,Licenses,Reminders,Notes,Short Title,Electronic location and access,ISBN (Normalized),ISBN_y,MMS Id,Title_y,Electronic Collection Public Name,Availability,Portfolio Id,Service Authentication Note,No. of Available Portfolio,Portfolio Public Note,Portfolio Authentication Note,Electronic location split,Electronic location,Electronic access,Concatenated Notes
0,The Lais of Marie de France : Text and Transla...,NaN,Book {Book - Physical} text; computer; online ...,"Waters, Claire M. (Orchard Park : Broadview Pr...",NaN,NaN,NaN,2026-05-27 09:15:00,NaN,NaN,NaN,unknown,2026-05-27 09:15:00,English,9781770486003,9781770486003,NaN,NaN,(VST)9781770486003,9926966302202311,72,NaN,NaN,NaN,841/.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Electronic version at VitalSource Bookshelf; B...,NaN,NaN,0,0,1 online resource,NaN,1,0,0,0,0,The Lais of Marie de France :,https://resolver.vitalsource.com/9781770486003,9781770486003; 1770486003; 9781554810826; 1554...,9781770486003; 1770486003; 9781554810826; 1554...,9926966302202311,The Lais of Marie de France : Text and Transla...,Broadview Editions Bookshelf,Available,53462931570002311,NaN,1,NaN,NaN,[https://resolver.vitalsource.com/9781770486003],https://resolver.vitalsource.com/9781770486003,,https://resolver.vitalsource.com/9781770486003
1,Red book : 2024-2027 report of the Committee o...,NaN,Book {Book - Electronic} text; computer; onlin...,American Academy of Pediatrics. Committee on I...,Communicable diseases in children--Periodicals...,STAT!Ref electronic medical library. Red Book:...,NaN,2026-04-03 04:22:00,33rd edition.,NaN,NaN,English,2026-04-02 10:08:00,NaN,9781610027359; 1610027353; 9781610027342,9781610027359; 9781610027359; 9781610027342,NaN,NaN,90103981814; (NhCcYBP)ybp306401256,9926931276402311,111,NaN,NaN,WC 100,618.929,RJ240 .A44 2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Electronic version at EBSCOhost; EBSCO Ebooks ...,NaN,NaN,1,0,"1 online resource (L, 1261 pages) :",NaN,1,0,0,0,0,Red book :,https://search.ebscohost.com/login.aspx?direct...,1610027353; 9781610027359; 9781610027342; 1610...,1610027353; 9781610027359; 9781610027342; 1610...,9926931276402311,Red book : 2024-2027 report of the Committee o...,EBSCO Ebooks Individually Purchased,Available,53454770480002311,NaN,1,NaN,Online Access - Unlimited users,[https://search.ebscohost.com/login.aspx?direc...,https://search.ebscohost.com/login.aspx?direct...,Online Access - Unlimited users,https://search.ebscohost.com/login.aspx?direct...
2,"Sales force management : leadership, innovatio...",NaN,Book {Book - Electronic} text; computer; onlin...,"Johnston, Mark W., (New York, NY : Routledge, ...",Sales management.,NaN,NaN,2026-03-27 13:24:00,Fourteenth edition.,NaN,NaN,English,2025-06-10 08:16:00,English,1-04-030131-2,9781040301319,NaN,NaN,(CKB)39206405800041; (CaSebORM)9781040301319; ...,9926916167202311,90,NaN,NaN,NaN,658.8/1,HF5438.4 .C48 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Electronic version at ProQuest; O'Reilly for H...,NaN,NaN,0,0,"1 online resource (xxx, 470 pages) :",NaN,0,0,0,0,0,Sales force management :,NaN,1040301312; 9781040301319,1040301312; 9781040301319,9926916167202311,"Sales force management : leadership, innovatio...",O'Reilly Online Learning: Academic/Public Libr...,Available,53454615920002311,NaN,1,NaN,NaN,NaN,NaN,NaN,
3,How great leaders think : the art of reframing,NaN,Book {Bo

In [24]:
#resultsPlusList.to_excel('Results Plus List Example.xlsx')

resultsPlusList.sort_values(by = 'Title_x')['Title_x'].unique()

array(['An evidence-based guide to assistive technology : meeting the needs of all students with disabilities / edited by Alice L. Rhodes and Victoria Slocum.',
       'Applied Artificial Intelligence in Business : Concepts and Cases.',
       "Beowulf : poem, poet and hero / Heather O'Donoghue.",
       'Child-centered play therapy : a practical guide to therapeutic relationships with children / Nancy H. Cochran, William J. Nordling, Jeff L. Cochran.',
       'Controversies in contemporary advertising [electronic resource] / Kim Bartel Sheehan.',
       'Core Competencies in Cognitive-Behavioral Therapy : Becoming a Highly Effective and Competent Cognitive-Behavioral Therapist / Cory F. Newman.',
       'Deaf culture : exploring deaf communities in the United States / Irene W. Leigh, PhD, Jean F. Andrews, PhD, Raychelle L. Harris, PhD, Topher González Ávila, MA.',
       'Demand and Supply Integration : The Key to World-Class Demand Forecasting, Second Edition.',
       'Ethics of dat

### Expand the ISBN Normalized Column

In [25]:
resultsPlusList['ISBN (Normalized) List'] = resultsPlusList['ISBN (Normalized)'].apply(lambda x: x.split(";"))

In [26]:
expandedList = resultsPlusList.explode('ISBN (Normalized) List')

expandedList['ISBN (Normalized) List'] = expandedList['ISBN (Normalized) List'].str.strip()

#expandedList.to_excel('Expanded List Example.xlsx')

In [27]:
finalJoin = pd.merge(expandedList, bookstoreList, left_on = 'ISBN (Normalized) List', right_on = 'ISBN', how = 'inner')

finalJoin = finalJoin.rename(columns = {'Short Title_y': 'Short Title'})

finalJoin

,Title_x,Alternate Graphic Representation,Type,Creator / Imprint,Subject,Series,Relation,Modification Date,Edition,Uniform Title,Medium Type,Language,Creation Date,Language of Cataloging,ISBN_x,ISBN (13),ISSN,Cluster ISSN,Record Number,MMS ID,Bibliographic Rank,Local Call Number,Other Classification Number,NLM Call Number,Dewey Decimal Classification Number,LC Call Number,Classification Part,UDC Classification Number,NAL Call Number,Handle Identifier,Peer Reviewed,Open Access,Physical Availability,Digital Availability,Electronic Availability,Collection Name,Collection ID,Orders,Requests,Physical Description Extent,Local Labels,Related Records,Courses,Licenses,Reminders,Notes,Short Title_x,Electronic location and access,ISBN (Normalized),ISBN_y,MMS Id,Title_y,Electronic Collection Public Name,Availability,Portfolio Id,Service Authentication Note,No. of Available Portfolio,Portfolio Public Note,Portfolio Authentication Note,Electronic location split,Electronic location,Electronic access,Concatenated Notes,ISBN (Normalized) List,Academic Term,Academic Department Name,Section Code,Instructor Name,ISBN,Short Title,Book Author,Book Publisher,Enrollment Capacity
0,The Lais of Marie de France : Text and Transla...,NaN,Book {Book - Physical} text; computer; online ...,"Waters, Claire M. (Orchard Park : Broadview Pr...",NaN,NaN,NaN,2026-05-27 09:15:00,NaN,NaN,NaN,unknown,2026-05-27 09:15:00,English,9781770486003,9781770486003,NaN,NaN,(VST)9781770486003,9926966302202311,72,NaN,NaN,NaN,841/.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Electronic version at VitalSource Bookshelf; B...,NaN,NaN,0,0,1 online resource,NaN,1,0,0,0,0,The Lais of Marie de France :,https://resolver.vitalsource.com/9781770486003,9781770486003; 1770486003; 9781554810826; 1554...,9781770486003; 1770486003; 9781554810826; 1554...,9926966302202311,The Lais of Marie de France : Text and Transla...,Broadview Editions Bookshelf,Available,53462931570002311,NaN,1,NaN,NaN,[https://resolver.vitalsource.com/9781770486003],https://resolver.vitalsource.com/9781770486003,,https://resolver.vitalsource.com/9781770486003,9781770486003,Summer 2026,ENGLISH,202630-ENGL.301.301-88080,"Perry, Ryan",9781770486003,THE LAIS OF MARIE DE FRANCE : TEXT AND TRANSLA...,NaN,BROADVIEW PRESS,15
1,The Lais of Marie de France : Text and Transla...,NaN,Book {Book - Physical} text; computer; online ...,"Waters, Claire M. (Orchard Park : Broadview Pr...",NaN,NaN,NaN,2026-05-27 09:15:00,NaN,NaN,NaN,unknown,2026-05-27 09:15:00,English,9781770486003,9781770486003,NaN,NaN,(VST)9781770486003,9926966302202311,72,NaN,NaN,NaN,841/.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Electronic version at VitalSource Bookshelf; B...,NaN,NaN,0,0,1 online resource,NaN,1,0,0,0,0,The Lais of Marie de France :,https://resolver.vitalsource.com/9781770486003,9781770486003; 1770486003; 9781554810826; 1554...,9781770486003; 1770486003; 9781554810826; 1554...,9926966302202311,The Lais of Marie de France : Text and Transla...,Broadview Editions Bookshelf,Available,53462931570002311,NaN,1,NaN,NaN,[https://resolver.vitalsource.com/9781770486003],https://resolver.vitalsource.com/9781770486003,,https://resolver.vitalsource.com/9781770486003,9781770486003,Summer 2026,ENGLISH,202630-ENGL.301.302-88081,"Perry, Ryan",9781770486003,THE LAIS OF MARIE DE FRANCE : TEXT AND TRANSLA...,NaN,BROADVIEW PRESS,10
2,Red book : 2024-2027 report of the Committee o...,NaN,Book {Book - Electronic} text; computer; onlin...,American Academy of Pediatrics. Committee on I...,Communicable diseases in children--Periodicals...,STAT!Ref electronic medical library. Red Book:...,NaN,2026-04-03 04:22:00,33rd edition.,NaN,NaN,English,2026-04-02 10:08:00,NaN,9781610027359; 1610027353; 9781610027342,9781610027359; 9781610027359; 9781610027342,NaN,NaN,90103981814; (NhCcYBP)ybp306401256,9926931276402311,111,NaN,NaN,WC 100,618.929,RJ240 .A44 2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Electronic version at EBSCOhost; EBSCO Ebooks ...,NaN,NaN,1,0,"1 online resource (L, 1261 pages) :",NaN,1,0,

In [28]:
selectiveCols = finalJoin[['Academic Term', 'Academic Department Name', 'Section Code', 'Instructor Name', 'ISBN (Normalized) List', 'Short Title', 'Book Author', 'Book Publisher', 'Enrollment Capacity', 'MMS ID', 'Electronic location', 'Electronic access', 'Electronic Availability']]

selectiveCols = selectiveCols.drop_duplicates()

selectiveCols = selectiveCols.rename(columns = {'Section Code': 'Course Code'})

len(selectiveCols.sort_values(by = 'Short Title')['Short Title'].unique())

49

In [29]:
selectiveCols.sort_values(by = 'Short Title')['Short Title'].unique()

array(['AN EVIDENCE-BASED GUIDE TO ASSISTIVE TECHNOLOGY : MEETING THE NEEDS OF ALL STUDENTS WITH DISABILITIES',
       'APPLIED ARTIFICIAL INTELLIGENCE IN BUSINESS',
       'BEOWULF A NEW VERSE TRANSLATION BILINGUAL EDITION',
       'CHILD-CENTERED PLAY THERAPY',
       'CONTROVERSIES IN CONTEMPORARY ADVERTISING',
       'CORE COMPETENCIES IN COGNITIVE-BEHAVIORAL THERAPY',
       'DEAF CULTURE : EXPLORING DEAF COMMUNITIES IN THE UNITED STATES',
       'DEMAND AND SUPPLY INTEGRATION : THE KEY TO WORLD-CLASS DEMAND FORECASTING, SECOND EDITION',
       'ETHICS OF DATA AND ANALYTICS',
       'ETHICS OF DATA AND ANALYTICS : CONCEPTS AND CASES', 'EVICTED',
       'EVIDENCE-BASED PRACTICE IMPROVEMENT : MERGING EVIDENCE-BASED PRACTICE AND QUALITY IMPROVEMENT',
       'GLOBAL WRITING FOR PUBLIC RELATIONS',
       'GUIDE TO THE VASCULAR PLANTS OF TENNESSEE',
       'HEY WHIPPLE, SQUEEZE THIS',
       'HOW GREAT LEADERS THINK : THE ART OF REFRAMING',
       'HUMAN OSTEOLOGY', 'INSIDE OUT AND OUTS

In [30]:
# Filter out the titles whose access is not Unlimited

salvagedCols = selectiveCols[(selectiveCols['Short Title'] == 'GUIDE TO THE VASCULAR PLANTS OF TENNESSEE')] #Use this to save titles from the delete codes that come later in this cell!

selectiveCols = selectiveCols[(selectiveCols['Electronic access'] != 'Online Access - 1 user') & (selectiveCols['Electronic access'] != 'Single User Single User') & (selectiveCols['Electronic access'] != 'Three User Three User') & (selectiveCols['Electronic access'] != 'Online Access - 3 users') & (selectiveCols['Electronic access'] != 'Online access : 2 simultaneous users; http://www.ebrary.com/corp/legal.jsp Click here for terms of use') & (selectiveCols['Electronic access'] != 'Online access - 1 simultaneous user') & (selectiveCols['Electronic access'] != 'Online Access - 3 simultaneous users.')]

selectiveCols = selectiveCols[(selectiveCols['Short Title'] != 'BEOWULF A NEW VERSE TRANSLATION BILINGUAL EDITION') & (selectiveCols['Short Title'] != 'CEREMONY') & (selectiveCols['Short Title'] != 'CRAFTING THE PERSONAL ESSAY : A GUIDE FOR WRITING AND PUBLISHING CREATIVE NON-FICTION') & (selectiveCols['Short Title'] != 'EVERYTHING IS OBVIOUS : HOW COMMON SENSE FAILS US') & (selectiveCols['Short Title'] != 'FUN HOME : A FAMILY TRAGICOMIC') & (selectiveCols['Short Title'] != 'INFORMATION ARCHITECTURE F/WORLD WI') & (selectiveCols['Short Title'] != 'ME AND MY DADDY LISTEN TO BOB MARLEY : NOVELLAS AND STORIES') & (selectiveCols['Short Title'] != 'PUBLIC LIBRARIES AND THEIR COMMUNITIES') & (selectiveCols['Short Title'] != 'STRATEGIC PLANNING FOR PUBLIC RELATIONS') & (selectiveCols['Short Title'] != 'SUN ALSO RISES') & (selectiveCols['Short Title'] != 'THE PROFESSOR IS IN : THE ESSENTIAL GUIDE TO TURNING YOUR PH. D. INTO A JOB') & (selectiveCols['Short Title'] != "ALL THAT SHE CARRIED : THE JOURNEY OF ASHLEY'S SACK, A BLACK FAMILY KEEPSAKE") & (selectiveCols['Short Title'] != 'INTERPERSONAL COMMUNICATION') & (selectiveCols['Short Title'] != 'LET NOBODY TURN US AROUND') & (selectiveCols['Short Title'] != 'LINEAR SYSTEMS THEORY') & (selectiveCols['Short Title'] != 'MERCY') & (selectiveCols['Short Title'] != 'NATURES BEST HOPE') & (selectiveCols['Short Title'] != 'FACILITATING BREAKTHROUGH : HOW TO REMOVE OBSTACLES, BRIDGE DIFFERENCES, AND MOVE FORWARD TOGETHER&NBSP;')]

selectiveCols = selectiveCols[selectiveCols['MMS ID'] != 9926793973502311]

selectiveCols = pd.concat([selectiveCols, salvagedCols])

selectiveCols

,Academic Term,Academic Department Name,Course Code,Instructor Name,ISBN (Normalized) List,Short Title,Book Author,Book Publisher,Enrollment Capacity,MMS ID,Electronic location,Electronic access,Electronic Availability
0,Summer 2026,ENGLISH,202630-ENGL.301.301-88080,"Perry, Ryan",9781770486003,THE LAIS OF MARIE DE FRANCE : TEXT AND TRANSLA...,NaN,BROADVIEW PRESS,15,9926966302202311,https://resolver.vitalsource.com/9781770486003,,Electronic version at VitalSource Bookshelf; B...
1,Summer 2026,ENGLISH,202630-ENGL.301.302-88081,"Perry, Ryan",9781770486003,THE LAIS OF MARIE DE FRANCE : TEXT AND TRANSLA...,NaN,BROADVIEW PRESS,10,9926966302202311,https://resolver.vitalsource.com/9781770486003,,Electronic version at VitalSource Bookshelf; B...
2,Summer 2026,NURSING,202630-NURS.670.001-85245,"Malone, Marian",9781610027359,RED BOOK 2024 : REPORT OF THE COMMITTEE ON INF...,NaN,AMERICAN ACADEMY OF PEDIATRICS,9,9926931276402311,https://search.ebscohost.com/login.aspx?direct...,Online Access - Unlimited users,Electronic version at EBSCOhost; EBSCO Ebooks ...
3,Summer 2026,NURSING,202630-NURS.670.002-87778,"Neal, Allyson",9781610027359,RED BOOK 2024 : REPORT OF THE COMMITTEE ON INF...,NaN,AMERICAN ACADEMY OF PEDIATRICS,5,9926931276402311,https://search.ebscohost.com/login.aspx?direct...,Online Access - Unlimited users,Electronic version at EBSCOhost; EBSCO Ebooks ...
4,Summer 2026,MARKETING,202630-MARK.471.301-88179,"Presley Bone, Blair",9781040301319,"SALES FORCE MANAGEMENT : LEADERSHIP, INNOVATIO...","JOHNSTON, MARK W., MARSHALL, GREG W., AND OGIL...","ROUTLEDGE, CHAPMAN & HALL, INCORPORATED",35,9926916167202311,NaN,NaN,Electronic version at ProQuest; O'Reilly for H...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,Summer 2026,SUPPLY CHAIN MANAGMENET,202630-SCM.547.002-85249,"Pellathy, Daniel",9781606492468,THE SUPPLY AND OPERATIONS MANGEMENT COLLECTION...,"FEIGIN, GERALD",BUSINESS EXPERT PRESS,30,9925725944102311,https://ebookcentral.proquest.com/lib/utk/deta...,Click to View,Electronic version at ProQuest; ProQuest Ebook...
143,Summer 2026,SOCIAL WORK,202630-SOWK.250.001-84600,"Mccormack, Sarah",9781483380094,SOCIAL WELFARE POLICY F/A SUSTAINABLE FUTURE,VAN WORMER,SAGE,15,9925617965602311,https://ebookcentral.proquest.com/lib/utk/deta...,Unlimited Access Unlimited Access,Electronic version at ProQuest; ProQuest Ebook...
144,Summer 2026,SOCIAL WORK,202630-SOWK.250.002-84791,"Mccormack, Sarah",9781483380094,SOCIAL WELFARE POLICY F/A SUSTAINABLE FUTURE,VAN WORMER,SAGE,15,9925617965602311,https://ebookcentral.proquest.com/lib/utk/deta...,Unlimited Access Unlimited Access,Electronic version at ProQuest; ProQuest Ebook...
147,Summer 2026,SUPPLY CHAIN MANAGMENET,202630-SCM.547.002-85249,"Pellathy, Daniel",9781606492468,THE SUPPLY AND OPERATIONS MANGEMENT COLLECTION...,"FEIGIN, GERALD",BUSINESS EXPERT PRESS,30,9924042300102311,https://search.ebscohost.com/login.aspx?direct...,,Electronic version at ProQuest; ProQuest Ebook...


In [31]:
len(selectiveCols.sort_values(by = 'Short Title')['Short Title'].unique())

41

In [32]:
nonUnlimited = finalJoin[~finalJoin['Short Title'].isin(selectiveCols['Short Title'])].dropna(how = 'all')

nonUnlimited.sort_values(by = 'Short Title')['Short Title'].unique()

array(['BEOWULF A NEW VERSE TRANSLATION BILINGUAL EDITION', 'EVICTED',
       'PROPERTY OF THE REBEL LIBRARIAN',
       'PURSUING HAPPINESS : A BEDFORD SPOTLIGHT READER',
       'RELIGIO MEDICI AND URNE-BURIALL', 'THE PORTABLE FEMINIST READER',
       'THE PRACTICE OF ADAPTIVE LEADERSHIP',
       'TRAUMA COUNSELING : THEORIES AND INTERVENTIONS FOR MANAGING TRAUMA, STRESS, CRISIS, AND DISASTER'],
      dtype=object)

## Export the Overlap Report

### This is the dataset that gets transferred to the Transforming Bookstore List Python code.

In [33]:
selectiveCols.to_excel(date_str+'_overlap_report.xlsx', index = False)

## Export the Bookstore Results to Send to the Bookstore

In [34]:
warnings.filterwarnings("ignore")

bookstoreExport = selectiveCols

# Fill in the Primo Link based on the MMS ID

for i in range(len(bookstoreExport['MMS ID'])):
    bookstoreExport['Electronic location'].iloc[i] = 'https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma' + str(bookstoreExport['MMS ID'].iloc[i])

bookstoreExport = bookstoreExport.rename(columns = {'ISBN (Normalized) List': 'ISBN', 'Electronic access': 'Access_Type', 'Electronic location': 'Permalink', 'Course Code': 'Section Code'})

bookstoreExport = bookstoreExport[['Academic Term', 'Academic Department Name', 'Section Code', 'Instructor Name', 'ISBN', 'Short Title', 'Book Author', 'Book Publisher', 'Enrollment Capacity', 'MMS ID', 'Access_Type', 'Electronic Availability', 'Permalink']]

bookstoreExport

,Academic Term,Academic Department Name,Section Code,Instructor Name,ISBN,Short Title,Book Author,Book Publisher,Enrollment Capacity,MMS ID,Access_Type,Electronic Availability,Permalink
0,Summer 2026,ENGLISH,202630-ENGL.301.301-88080,"Perry, Ryan",9781770486003,THE LAIS OF MARIE DE FRANCE : TEXT AND TRANSLA...,NaN,BROADVIEW PRESS,15,9926966302202311,,Electronic version at VitalSource Bookshelf; B...,https://utk.primo.exlibrisgroup.com/permalink/...
1,Summer 2026,ENGLISH,202630-ENGL.301.302-88081,"Perry, Ryan",9781770486003,THE LAIS OF MARIE DE FRANCE : TEXT AND TRANSLA...,NaN,BROADVIEW PRESS,10,9926966302202311,,Electronic version at VitalSource Bookshelf; B...,https://utk.primo.exlibrisgroup.com/permalink/...
2,Summer 2026,NURSING,202630-NURS.670.001-85245,"Malone, Marian",9781610027359,RED BOOK 2024 : REPORT OF THE COMMITTEE ON INF...,NaN,AMERICAN ACADEMY OF PEDIATRICS,9,9926931276402311,Online Access - Unlimited users,Electronic version at EBSCOhost; EBSCO Ebooks ...,https://utk.primo.exlibrisgroup.com/permalink/...
3,Summer 2026,NURSING,202630-NURS.670.002-87778,"Neal, Allyson",9781610027359,RED BOOK 2024 : REPORT OF THE COMMITTEE ON INF...,NaN,AMERICAN ACADEMY OF PEDIATRICS,5,9926931276402311,Online Access - Unlimited users,Electronic version at EBSCOhost; EBSCO Ebooks ...,https://utk.primo.exlibrisgroup.com/permalink/...
4,Summer 2026,MARKETING,202630-MARK.471.301-88179,"Presley Bone, Blair",9781040301319,"SALES FORCE MANAGEMENT : LEADERSHIP, INNOVATIO...","JOHNSTON, MARK W., MARSHALL, GREG W., AND OGIL...","ROUTLEDGE, CHAPMAN & HALL, INCORPORATED",35,9926916167202311,NaN,Electronic version at ProQuest; O'Reilly for H...,https://utk.primo.exlibrisgroup.com/permalink/...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,Summer 2026,SUPPLY CHAIN MANAGMENET,202630-SCM.547.002-85249,"Pellathy, Daniel",9781606492468,THE SUPPLY AND OPERATIONS MANGEMENT COLLECTION...,"FEIGIN, GERALD",BUSINESS EXPERT PRESS,30,9925725944102311,Click to View,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...
143,Summer 2026,SOCIAL WORK,202630-SOWK.250.001-84600,"Mccormack, Sarah",9781483380094,SOCIAL WELFARE POLICY F/A SUSTAINABLE FUTURE,VAN WORMER,SAGE,15,9925617965602311,Unlimited Access Unlimited Access,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...
144,Summer 2026,SOCIAL WORK,202630-SOWK.250.002-84791,"Mccormack, Sarah",9781483380094,SOCIAL WELFARE POLICY F/A SUSTAINABLE FUTURE,VAN WORMER,SAGE,15,9925617965602311,Unlimited Access Unlimited Access,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...
147,Summer 2026,SUPPLY CHAIN MANAGMENET,202630-SCM.547.002-85249,"Pellathy, Daniel",9781606492468,THE SUPPLY AND OPERATIONS MANGEMENT COLLECTION...,"FEIGIN, GERALD",BUSINESS EXPERT PRESS,30,9924042300102311,,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...


In [35]:
bookstoreExport.to_excel(date_str+'BookStoreResults_TPQ.xlsx', index = False) #Send this to the Bookstore